# InferenceEngine do Alakoro FiberSense

> Motor de inferência canônica para análise de fibras ópticas distribuídas (DTS/DAS/DSS) em poços de petróleo e gás.

Este notebook aprofunda a implementação, o uso e a extensão do **InferenceEngine** do Alakoro FiberSense. O motor é escrito em **C++20** (`src/cpp/include/alakoro/inference_engine.hpp`), exposto ao Python via **pybind11** e envolvido por uma camada Python (`src/ontology/inference_engine.py`) que converte os resultados em instâncias da ontologia Alakoro.

Os exemplos de código são executáveis no ambiente atual do projeto.

In [ ]:
# Configuração inicial e imports usados ao longo do notebook
import sys
import json
import os

import numpy as np
import matplotlib.pyplot as plt

# Módulo C++ compilado (pybind11)
from alakoro_core import (
    InferenceResult,
    InferenceMetadata,
    CanonicalInferenceEngine,
)

# Wrapper Python + ontologia
from src.ontology import InferenceEngine, infer_events
from src.ontology.events import (
    Event,
    JouleThomsonEvent,
    LeakEvent,
    FlowEvent,
    WarmBackEvent,
)

# Gerador de assinaturas sintéticas
from src.simulation import SignatureGenerator, WellGeometry, AcquisitionConfig

print("Ambiente OK. Python:", sys.executable)
print("CanonicalInferenceEngine:", CanonicalInferenceEngine)


## 1. O que é o InferenceEngine?

O **InferenceEngine** é o componente responsável por transformar matrizes de aquisição `(n_times, n_channels)` em uma lista estruturada de hipóteses sobre eventos operacionais em poços de petróleo.

Para cada hipótese, ele emite:

- `event_type`: código canônico (ex: `"joule_thomson"`).
- `event_label_pt` / `event_label_en`: rótulos em português e inglês.
- `confidence`: escore entre `0.0` e `1.0`.
- `depth_md`: profundidade estimada (m).
- `severity`: `"Low"`, `"Medium"` ou `"High"`.
- `recommendation`: ação operacional sugerida.

Os 15 eventos canônicos reconhecidos são:

| # | Código | Rótulo em português |
|---|--------|---------------------|
| 1 | `joule_thomson` | Dipolo Térmico Joule-Thomson |
| 2 | `slope_velocity` | Rastreamento de Inclinação (Velocidade) |
| 3 | `warm_back` | Recuperação Térmica (Warm-Back) |
| 4 | `valve_chatter` | Chatter/Multipointing de Válvula |
| 5 | `slugging_cycle` | Ciclo de Slugging |
| 6 | `leak_path` | Caminho de Vazamento Tubing-Ânulo |
| 7 | `glv_bellow_rupture` | Fole Furado de Válvula de Gás Lift |
| 8 | `perforation_effectiveness` | Efetividade de Canhoneio |
| 9 | `frac_screenout` | Embuchamento de Fratura (Screen-out) |
| 10 | `frac_proppant_distribution` | Distribuição de Propante |
| 11 | `frac_height_growth` | Crescimento de Altura de Fratura |
| 12 | `cement_bond_evaluation` | Avaliação de Cimentação (CBL/VDL) |
| 13 | `re_cementing_assessment` | Avaliação de Recimentação |
| 14 | `crossflow_zonal` | Fluxo Cruzado Zonal |
| 15 | `cement_channeling` | Canalização de Cimento |

In [ ]:
# Enumera os 15 eventos consultando os traits compilados no C++
# (usamos o wrapper Python para obter os rótulos via inferência de uma matriz nula)
meta = InferenceMetadata()
meta.sampling_rate_hz = 1000.0
meta.depth_step_m = 1.0
meta.surface_temp_c = 20.0
meta.geo_gradient_cpm = 0.03

engine = CanonicalInferenceEngine()
dts_null = np.zeros((10, 3000), dtype=np.float64)
results = engine.infer(dts_null, None, meta)
print("Eventos detectados em matriz nula:", len(results))
for r in results:
    print(f"  {r.event_type:35s} | {r.event_label_pt}")


## 2. Arquitetura geral

### 2.1 Entrada

| Fonte | Tipo | Obrigatoriedade | Descrição |
|-------|------|-----------------|-----------|
| `dts` | `np.ndarray` 2D | obrigatório | Matriz de temperatura `(n_times, n_channels)`. |
| `das` | `np.ndarray` 2D | opcional | Matriz acústica com o mesmo shape de `dts`. |
| `metadata` | `InferenceMetadata` | obrigatório | Parâmetros de aquisição e contexto geotérmico. |

### 2.2 Saída

Cada regra emite zero ou mais `InferenceResult`. Abaixo vemos a estrutura de um resultado no Python.

### 2.3 Pipeline

```
DTS (obrig.) + DAS (opc.) + InferenceMetadata
        |
        v
Pré-processamento (temporal_mean, remove_polynomial_baseline,
                   remove_median_baseline, adaptive_threshold)
        |
        v
Regras canônicas (uma struct por evento, cada apply retorna ResultGenerator)
        |
        v
Agregação (fold expression executa todas as regras;
           collect_results consome os generators)
        |
        v
Binding Python (pybind11) + wrapper Python (converte para Event)
```

O pipeline é **síncrono do ponto de vista do Python**: o C++ consome internamente os generators corrotinados e devolve um `std::vector<InferenceResult>` já materializado.

In [ ]:
# Inspeciona a estrutura de InferenceResult e InferenceMetadata
print("InferenceResult atributos:")
print("  event_type      :", type(InferenceResult.event_type).__name__)
print("  event_label_pt  :", type(InferenceResult.event_label_pt).__name__)
print("  event_label_en  :", type(InferenceResult.event_label_en).__name__)
print("  confidence      :", type(InferenceResult.confidence).__name__)
print("  depth_md        :", type(InferenceResult.depth_md).__name__)
print("  severity        :", type(InferenceResult.severity).__name__)
print("  recommendation  :", type(InferenceResult.recommendation).__name__)

m = InferenceMetadata()
print("\nInferenceMetadata defaults:")
print("  sampling_rate_hz :", m.sampling_rate_hz)
print("  depth_step_m     :", m.depth_step_m)
print("  surface_temp_c   :", m.surface_temp_c)
print("  geo_gradient_cpm :", m.geo_gradient_cpm)


## 3. Implementação C++20

O arquivo `src/cpp/include/alakoro/inference_engine.hpp` concentra toda a lógica canônica. Os elementos centrais são:

- `enum class CanonicalEvent`: 15 tags fortes para os eventos.
- `EventTraits<E>`: metaprogramação com `constexpr std::string_view` para nome, rótulos e recomendação em tempo de compilação.
- `ResultGenerator`: corrotina C++20 mínima que produz `InferenceResult` via `co_yield`.
- `InferenceRule`: concept que exige `static ResultGenerator apply(...)`.
- `InferenceEngine<Events...>`: template variádico que usa `if constexpr` e fold expressions para executar todas as regras.
- `CanonicalInferenceEngine`: alias com todos os 15 eventos.

A seguir, exemplificamos os conceitos via código Python que replica mentalmente o C++.

### 3.1 `CanonicalEvent` e `EventTraits<E>`

O enum forte evita conversões implícitas. `EventTraits<E>` é uma especialização de template que associa strings a cada evento em tempo de compilação:

```cpp
enum class CanonicalEvent : std::uint8_t {
    JouleThomson, SlopeVelocity, WarmBack, ValveChatter, SluggingCycle,
    LeakPath, GlvBellowRupture, PerforationEffectiveness, FracScreenout,
    FracProppantDistribution, FracHeightGrowth, CementBondEvaluation,
    ReCementingAssessment, CrossflowZonal, CementChanneling
};

#define ALAKORO_EVENT_TRAITS(EVENT, CODE, PT, EN, RECO) \\
    template <>                                          \\
    struct EventTraits<CanonicalEvent::EVENT> {          \\
        static constexpr std::string_view code = CODE;   \\
        static constexpr std::string_view label_pt = PT; \\
        static constexpr std::string_view label_en = EN; \\
        static constexpr std::string_view recommendation = RECO; \\
    }
```

A função `make_result<E>` usa esses traits para preencher automaticamente os campos textuais.

In [ ]:
# Equivalente conceitual em Python (traits de evento)
_EVENT_TRAITS = {
    "joule_thomson": {
        "label_pt": "Dipolo Térmico Joule-Thomson",
        "label_en": "Joule-Thomson Thermal Dipole",
        "recommendation": "Verificar passagem de gas/líquido na interface e validar PVT local.",
    },
    "glv_bellow_rupture": {
        "label_pt": "Fole Furado de Válvula de Gás Lift",
        "label_en": "Gas Lift Valve Bellow Rupture",
        "recommendation": "Substituir ou reparar o GLV com fole comprometido.",
    },
    "cement_bond_evaluation": {
        "label_pt": "Avaliação de Cimentação (CBL/VDL)",
        "label_en": "Cement Bond Evaluation (CBL/VDL)",
        "recommendation": "Correlacionar com log CBL/VDL para mapear qualidade do cimento.",
    },
}

for code, traits in _EVENT_TRAITS.items():
    print(f"{code} -> {traits['label_pt']}")


### 3.2 `ResultGenerator` — corrotinas C++20

Cada regra é uma corrotina. A implementação é um generator mínimo, sem dependências externas:

```cpp
struct ResultGenerator {
    struct promise_type {
        InferenceResult current_value;
        ResultGenerator get_return_object() { ... }
        std::suspend_always initial_suspend() noexcept { return {}; }
        std::suspend_always final_suspend() noexcept { return {}; }
        void unhandled_exception() { std::terminate(); }
        void return_void() noexcept {}
        std::suspend_always yield_value(InferenceResult value) noexcept {
            current_value = std::move(value);
            return {};
        }
    };
    // move-only; destrói o handle no destruidor
};
```

Uma regra típica:

```cpp
struct JouleThomsonRule {
    static ResultGenerator apply(...) {
        // heurística ...
        co_yield make_result<CanonicalEvent::JouleThomson>(conf, depth, severity);
        co_return;
    }
};
```

In [ ]:
# Equivalente minimalista de ResultGenerator em Python
from dataclasses import dataclass
from typing import Optional, Iterator

@dataclass
class PyInferenceResult:
    event_type: str
    label_pt: str
    confidence: float
    depth_md: float

class PyResultGenerator:
    """Generator simplificado que simula co_yield."""
    def __init__(self, items: list[PyInferenceResult]):
        self._items = items
        self._idx = -1

    def __iter__(self):
        return iter(self._items)

# Regra exemplo
class PyJouleThomsonRule:
    @staticmethod
    def apply() -> PyResultGenerator:
        return PyResultGenerator([
            PyInferenceResult("joule_thomson", "Dipolo Térmico Joule-Thomson", 0.85, 1500.0),
        ])

print("Resultados da regra Python:")
for r in PyJouleThomsonRule.apply():
    print(f"  {r.label_pt} @ {r.depth_md} m (conf={r.confidence})")


### 3.3 `InferenceEngine<Events...>` — template variádico

A engine é parametrizada por uma lista de `CanonicalEvent`. Internamente:

```cpp
template <CanonicalEvent... Events>
class InferenceEngine {
public:
    std::vector<InferenceResult> infer(...) const {
        std::vector<InferenceResult> all;
        all.reserve(sizeof...(Events));
        (execute_rule<Events>(dts, das, n_times, n_channels, meta, all), ...);
        return all;
    }
private:
    template <CanonicalEvent E>
    void execute_rule(...) const {
        ResultGenerator gen = [&]() {
            if constexpr (E == CanonicalEvent::JouleThomson) {
                return JouleThomsonRule::apply(...);
            } else if constexpr (E == CanonicalEvent::SlopeVelocity) {
                return SlopeVelocityRule::apply(...);
            }
            // ... todos os 15 eventos ...
        }();
        auto partial = collect_results(std::move(gen));
        out.insert(out.end(), std::make_move_iterator(partial.begin()),
                   std::make_move_iterator(partial.end()));
    }
};
```

A fold expression `(execute_rule<Events>(...), ...)` expande, em tempo de compilação, para uma chamada sequencial de cada regra.

In [ ]:
# Simulação da fold expression em Python
class PyRule:
    def __init__(self, name):
        self.name = name
    def apply(self):
        return [f"result:{self.name}"]

class PyInferenceEngine:
    def __init__(self, rules):
        self.rules = rules
    def infer(self):
        out = []
        # Equivalente a: (execute_rule<R>(), ...)
        for rule in self.rules:
            out.extend(rule.apply())
        return out

rules = [PyRule("joule_thomson"), PyRule("warm_back"), PyRule("leak_path")]
engine = PyInferenceEngine(rules)
print("Resultados agregados:", engine.infer())


## 4. Helpers numéricos

Todos os helpers residem em `alakoro::inference::detail`.

### 4.1 `temporal_mean`

Calcula o perfil médio ao longo do tempo para cada canal. A matriz está em layout row-major com índice linear `data[t * n_channels + c]`.

### 4.2 `remove_polynomial_baseline`

Ajusta um polinômio de grau `N` por mínimos quadrados e subtrai do perfil. O sistema normal `A^T A x = A^T b` é resolvido por eliminação de Gauss com pivoteamento parcial.

### 4.3 `remove_median_baseline`

Subtrai uma baseline de mediana móvel. A mediana é robusta a outliers.

### 4.4 `adaptive_threshold` (MAD / IQR)

Threshold robusto baseado em MAD (fator `1.4826`) ou IQR.

### 4.5 `find_peaks` / `find_valleys`

Detectam máximos e mínimos locais com amplitude mínima e distância mínima.

### 4.6 `das_energy_profile`

Calcula a energia quadrática média do DAS ao longo do tempo para cada canal.

In [ ]:
# Demonstração dos helpers equivalentes em Python
from scipy.signal import find_peaks as scipy_find_peaks

def temporal_mean(data, n_times, n_channels):
    return np.mean(data.reshape(n_times, n_channels), axis=0)

def percentile(v, p):
    return np.percentile(v, p)

def mad(v):
    med = percentile(v, 50)
    return percentile(np.abs(np.asarray(v) - med), 50)

def adaptive_threshold(v, method="mad", k=2.0):
    v = np.asarray(v)
    if method == "iqr":
        return k * (percentile(v, 75) - percentile(v, 25))
    return k * 1.4826 * mad(v)

def das_energy_profile(data, n_times, n_channels):
    data = data.reshape(n_times, n_channels)
    return np.mean(data**2, axis=0)

# Cria um perfil de teste
np.random.seed(42)
x = np.linspace(0, 3000, 3000)
profile = 20 + 0.03 * x  # baseline geotérmica
profile += -5 * np.exp(-0.5 * ((x - 1500) / 50) ** 2)
profile += 4 * np.exp(-0.5 * ((x - 1550) / 40) ** 2)
profile += np.random.normal(0, 0.1, size=x.shape)

anomaly_poly = profile - np.polyval(np.polyfit(np.arange(len(profile)), profile, 2), np.arange(len(profile)))
thr = adaptive_threshold(anomaly_poly, "mad", 1.5)
peaks, _ = scipy_find_peaks(anomaly_poly, height=thr, distance=30)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(x, profile, label="perfil médio")
axes[0].plot(x, 20 + 0.03 * x, label="baseline geotérmica", linestyle="--")
axes[0].set_ylabel("Temperatura (°C)")
axes[0].legend()
axes[1].plot(x, anomaly_poly, label="anomalia (baseline polinomial removida)")
axes[1].axhline(thr, color="r", linestyle="--", label=f"threshold = {thr:.2f}")
axes[1].scatter(x[peaks], anomaly_poly[peaks], color="green", zorder=5, label="picos")
axes[1].set_xlabel("Profundidade (m)")
axes[1].set_ylabel("Anomalia (°C)")
axes[1].legend()
plt.suptitle("Helpers numéricos: temporal_mean, baseline polinomial, adaptive_threshold, find_peaks")
plt.tight_layout()
plt.show()


## 5. Exemplos de regras detalhados

A seguir, analisamos três regras representativas e testamos cada uma com assinaturas sintéticas.

### 5.1 `JouleThomsonRule` — Dipolo Térmico Joule-Thomson

**Fenômeno físico**: quando um fluido atravessa uma restrição (interface de fases, estrangulamento), a expansão Joule-Thomson produz resfriamento seguido de aquecimento, formando um dipolo térmico no perfil de temperatura.

**Heurística implementada**:

1. Calcula o perfil médio temporal (`temporal_mean`).
2. Remove baseline polinomial de grau 2 (`remove_polynomial_baseline`).
3. Calcula threshold adaptativo com MAD e fator `k = 1.5`.
4. Percorre o perfil de anomalia com uma janela deslizante de tamanho `anomaly.size() / 20` (mínimo 5 amostras).
5. Para cada posição `i`, computa a média da janela anterior (`before`) e da janela posterior (`after`). O escore é `after - before`.
6. Se o melhor escore superar o threshold, emite um resultado na profundidade do ponto de maior contraste.

```cpp
if (best_score > threshold) {
    double depth = detail::channel_to_depth(best_idx, meta.depth_step_m);
    double conf = std::min(best_score / (5.0 * threshold), 1.0);
    co_yield make_result<CanonicalEvent::JouleThomson>(
        conf, depth, conf > 0.7 ? "High" : (conf > 0.4 ? "Medium" : "Low"));
}
```

In [ ]:
# Gera e visualiza uma assinatura Joule-Thomson
well = WellGeometry(depth_top=0, depth_bottom=3000, n_channels=3000)
acq = AcquisitionConfig(sampling_rate_hz=1000, trace_interval_s=2.0, duration_s=120)
gen = SignatureGenerator(well, acq)

sig_jt = gen.generate_joule_thomson(interface_depth=1500.0)
dts_jt = sig_jt["dts"]
das_jt = sig_jt["das"]

depth = gen.depth
mean_profile = np.mean(dts_jt, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im0 = axes[0].imshow(dts_jt.T, aspect="auto", origin="lower",
                     extent=[0, dts_jt.shape[0], depth[0], depth[-1]], cmap="RdBu_r")
axes[0].set_xlabel("Amostra temporal")
axes[0].set_ylabel("Profundidade (m)")
axes[0].set_title("DTS - Joule-Thomson")
fig.colorbar(im0, ax=axes[0], label="°C")

axes[1].plot(depth, mean_profile, label="perfil médio")
axes[1].axvline(1500, color="k", linestyle="--", label="interface esperada")
axes[1].set_xlabel("Profundidade (m)")
axes[1].set_ylabel("Temperatura média (°C)")
axes[1].set_title("Perfil médio temporal")
axes[1].legend()
plt.tight_layout()
plt.show()

# Inferência
engine = InferenceEngine()
events_jt = engine.infer_from_signature(sig_jt)
print("Eventos detectados na assinatura Joule-Thomson:")
for e in events_jt:
    print(f"  {e.event_type:35s} | depth={e.depth_md:8.1f} m | conf={e.confidence:.3f} | {e.severity}")


### 5.2 `GlvBellowRuptureRule` — Fole Furado de Válvula de Gás Lift

**Fenômeno físico**: válvulas de gás-lift instaladas em mandris ao longo do poço aparecem como picos acústicos espaçados regularmente. Quando o fole de uma válvula se rompe, a válvula para de operar e seu pico desaparece, criando um "gap" na sequência regular.

**Heurística implementada**:

1. Se houver DAS, calcula o perfil de energia (`das_energy_profile`) e filtra picos acima do percentil 90, com distância mínima de 30 canais.
2. Se o DAS for insuficiente (< 3 picos), usa o DTS como fallback.
3. Calcula os espaçamentos entre picos consecutivos.
4. Identifica um gap maior que `1.5 × mediana` dos espaçamentos.
5. Estima a profundidade de ruptura como a profundidade do pico anterior mais a mediana do espaçamento.

In [ ]:
# Gera e visualiza uma assinatura de ruptura de fole GLV
sig_glv = gen.generate_glv_bellow_rupture(
    valve_depth=1400.0, n_valves=5, rupture_valve_idx=2
)
dts_glv = sig_glv["dts"]
das_glv = sig_glv["das"]
energy_glv = np.mean(das_glv ** 2, axis=0)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
im1 = axes[0].imshow(das_glv.T, aspect="auto", origin="lower",
                     extent=[0, das_glv.shape[0], depth[0], depth[-1]], cmap="hot")
axes[0].set_ylabel("Profundidade (m)")
axes[0].set_title("DAS - GLV Bellow Rupture")
fig.colorbar(im1, ax=axes[0])

axes[1].plot(depth, energy_glv, label="energia média DAS")
axes[1].set_xlabel("Profundidade (m)")
axes[1].set_ylabel("Energia")
axes[1].set_title("Perfil de energia DAS")
axes[1].legend()
plt.tight_layout()
plt.show()

events_glv = engine.infer_from_signature(sig_glv)
print("Eventos detectados na assinatura GLV Bellow Rupture:")
for e in events_glv:
    print(f"  {e.event_type:35s} | depth={e.depth_md:8.1f} m | conf={e.confidence:.3f} | {e.severity}")


### 5.3 `CementBondEvaluationRule` — Avaliação de Cimentação

**Fenômeno físico**: zonas com bom cimento apresentam condutividade térmica mais uniforme, enquanto canais ou deslocamentos criam variações espaciais acentuadas no perfil de temperatura.

**Heurística implementada**:

1. Calcula o perfil médio temporal.
2. Remove baseline por mediana móvel com janela 51 (preserva anomalias localizadas).
3. Calcula o desvio padrão amostral do perfil de anomalia.
4. Se a dispersão for suficiente (`conf > 0.12`), emite um resultado na profundidade média do poço.

In [ ]:
# Gera e visualiza uma assinatura de avaliação de cimentação
sig_cem = gen.generate_cement_bond_evaluation()
dts_cem = sig_cem["dts"]
mean_profile_cem = np.mean(dts_cem, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im2 = axes[0].imshow(dts_cem.T, aspect="auto", origin="lower",
                     extent=[0, dts_cem.shape[0], depth[0], depth[-1]], cmap="RdBu_r")
axes[0].set_xlabel("Amostra temporal")
axes[0].set_ylabel("Profundidade (m)")
axes[0].set_title("DTS - Cement Bond Evaluation")
fig.colorbar(im2, ax=axes[0])

axes[1].plot(depth, mean_profile_cem)
axes[1].set_xlabel("Profundidade (m)")
axes[1].set_ylabel("Temperatura média (°C)")
axes[1].set_title("Perfil médio - variações espaciais de cimentação")
plt.tight_layout()
plt.show()

events_cem = engine.infer_from_signature(sig_cem)
print("Eventos detectados na assinatura Cement Bond Evaluation:")
for e in events_cem:
    print(f"  {e.event_type:35s} | depth={e.depth_md:8.1f} m | conf={e.confidence:.3f} | {e.severity}")


## 6. Integração Python

### 6.1 Bindings pybind11

O arquivo `src/cpp/src/bindings.cpp` expõe as seguintes entidades no módulo `_alakoro_core`:

- `InferenceResult` (read-only fields).
- `InferenceMetadata`.
- `CanonicalInferenceEngine` com método `infer(dts, das=None, metadata)`.
- Função helper `infer_events_d(dts, das=None, metadata)`.

### 6.2 Wrapper Python

A classe `InferenceEngine` em `src/ontology/inference_engine.py` converte os resultados C++ em objetos da ontologia Alakoro. Ela faz a ponte com o `SignatureGenerator` através do método `infer_from_signature`.

O mapeamento `_EVENT_CLASS_MAP` liga códigos canônicos a classes especializadas:

```python
_EVENT_CLASS_MAP = {
    "joule_thomson": JouleThomsonEvent,
    "slope_velocity": FlowEvent,
    "warm_back": WarmBackEvent,
    "leak_path": LeakEvent,
    # ... demais eventos mapeados para Event
}
```

In [ ]:
# Demonstração do uso direto do binding C++
cpp_engine = CanonicalInferenceEngine()
cpp_meta = InferenceMetadata()
cpp_meta.sampling_rate_hz = 1000.0
cpp_meta.depth_step_m = 1.0
cpp_meta.surface_temp_c = 20.0
cpp_meta.geo_gradient_cpm = 0.03

cpp_results = cpp_engine.infer(dts_jt, None, cpp_meta)
print("Resultados C++ brutos:")
for r in cpp_results:
    print(f"  {r.event_type:35s} | depth={r.depth_md:8.1f} m | conf={r.confidence:.3f}")

print("\nTipos de objetos retornados:")
print("  C++ :", type(cpp_results[0]).__name__)
print("  Ontologia:", type(events_jt[0]).__name__)


In [ ]:
# Inspeciona o objeto da ontologia
jt_event = next(e for e in events_jt if e.event_type == "joule_thomson")
print("Classe:", type(jt_event).__name__)
print("event_type     :", jt_event.event_type)
print("name           :", jt_event.name)
print("depth_md       :", jt_event.depth_md)
print("interface_depth:", jt_event.interface_depth)
print("confidence     :", jt_event.confidence)
print("severity       :", jt_event.severity)
print("recommendation :", jt_event.recommendation)


## 7. Exemplos práticos em Python

### 7.1 Inferir a partir de arrays NumPy

Podemos usar diretamente o binding C++ ou o wrapper Python.

In [ ]:
# Exemplo 1: criar um DTS sintético manualmente e inferir
n_times = 120
n_channels = 3000
depths = np.arange(n_channels) * 1.0

# Baseline geotérmica + dipolo Joule-Thomson em 1500 m
temps = 20.0 + 0.03 * depths
dipole = -5.0 * np.exp(-0.5 * ((depths - 1500.0) / 50.0) ** 2)
dipole += 4.0 * np.exp(-0.5 * ((depths - 1550.0) / 40.0) ** 2)
dts_synthetic = temps + dipole + np.random.normal(0, 0.1, (n_times, n_channels))

events_synthetic = infer_events(
    dts_synthetic,
    sampling_rate_hz=1000.0,
    depth_step_m=1.0,
    surface_temp_c=20.0,
    geo_gradient_cpm=0.03,
)

print("Eventos inferidos a partir de DTS sintético:")
for e in events_synthetic:
    print(f"  {e.event_type:35s} | depth={e.depth_md:8.1f} m | conf={e.confidence:.3f} | {e.severity}")


### 7.2 Inferir a partir de uma assinatura gerada pelo SignatureGenerator

O `SignatureGenerator` produz 15 assinaturas canônicas. O método `infer_from_signature` abstrai a extração dos parâmetros de aquisição.

In [ ]:
# Exemplo 2: usar SignatureGenerator + InferenceEngine
well2 = WellGeometry(depth_top=0, depth_bottom=3000, n_channels=3000)
acq2 = AcquisitionConfig(sampling_rate_hz=1000, trace_interval_s=2.0, duration_s=120)
gen2 = SignatureGenerator(well2, acq2)

sigs = {
    "joule_thomson": gen2.generate_joule_thomson(interface_depth=1500.0),
    "warm_back": gen2.generate_warm_back(injection_depths=[1200.0, 1500.0, 1800.0]),
    "leak_path": gen2.generate_leak_path(leak_depth=1914.0),
    "glv_bellow_rupture": gen2.generate_glv_bellow_rupture(),
    "cement_bond_evaluation": gen2.generate_cement_bond_evaluation(),
}

engine2 = InferenceEngine()
summary = []
for name, sig in sigs.items():
    events = engine2.infer_from_signature(sig)
    detected = [e.event_type for e in events]
    summary.append((name, detected))
    print(f"{name:35s} -> {detected}")

# Verifica se o evento esperado foi detectado
print("\nVerificação de detecção principal:")
for expected, (_, detected) in zip(sigs.keys(), summary):
    ok = expected in detected
    print(f"  {expected:35s} {'OK' if ok else 'FALHOU'} (detected={expected in detected})")


### 7.3 Rodar todas as 15 assinaturas e verificar cobertura

Este teste equivale ao caso de teste `test_all_canonical_events_at_least_once`.

In [ ]:
# Gera todas as 15 assinaturas e coleta todos os eventos detectados
generators = [
    gen2.generate_joule_thomson,
    gen2.generate_slope_velocity,
    gen2.generate_warm_back,
    gen2.generate_valve_chatter,
    gen2.generate_slugging_cycle,
    gen2.generate_leak_path,
    gen2.generate_glv_bellow_rupture,
    gen2.generate_perforation_effectiveness,
    gen2.generate_frac_screenout,
    gen2.generate_frac_proppant_distribution,
    gen2.generate_frac_height_growth,
    gen2.generate_cement_bond_evaluation,
    gen2.generate_re_cementing_assessment,
    gen2.generate_crossflow_zonal,
    gen2.generate_cement_channeling,
]

detected = set()
for gen_func in generators:
    events = engine2.infer_from_signature(gen_func())
    detected.update(e.event_type for e in events)

expected = {
    "joule_thomson", "slope_velocity", "warm_back", "valve_chatter",
    "slugging_cycle", "leak_path", "glv_bellow_rupture",
    "perforation_effectiveness", "frac_screenout", "frac_proppant_distribution",
    "frac_height_growth", "cement_bond_evaluation", "re_cementing_assessment",
    "crossflow_zonal", "cement_channeling",
}

missing = expected - detected
print(f"Eventos esperados: {len(expected)}")
print(f"Eventos detectados: {len(detected)}")
print(f"Faltando: {missing if missing else 'nenhum'}")
assert not missing, f"Eventos não detectados: {missing}"
print("\nCobertura completa dos 15 eventos canônicos confirmada!")


## 8. Como adicionar um novo evento

Para estender o motor com um novo evento canônico, siga os passos abaixo.

### Passo 1 — Adicionar o valor ao enum

```cpp
// src/cpp/include/alakoro/inference_engine.hpp
enum class CanonicalEvent : std::uint8_t {
    // ... eventos existentes ...
    NewEvent,
};
```

### Passo 2 — Definir os traits

```cpp
ALAKORO_EVENT_TRAITS(NewEvent,
    "new_event",
    "Novo Evento",
    "New Event",
    "Recomendação operacional para o novo evento.");
```

### Passo 3 — Implementar a regra

```cpp
struct NewEventRule {
    static ResultGenerator apply(std::span<const double> dts,
                                 std::span<const double> das,
                                 std::size_t n_times,
                                 std::size_t n_channels,
                                 const InferenceMetadata& meta) {
        // heurística ...
        co_yield make_result<CanonicalEvent::NewEvent>(conf, depth, severity);
        co_return;
    }
};
```

### Passo 4 — Registrar na engine

```cpp
template <CanonicalEvent E>
void execute_rule(...) const {
    ResultGenerator gen = [&]() {
        // ... ramos existentes ...
        } else if constexpr (E == CanonicalEvent::NewEvent) {
            return NewEventRule::apply(dts, das, n_times, n_channels, meta);
        }
    }();
    // ...
}

using CanonicalInferenceEngine = InferenceEngine<
    // ... eventos existentes ...
    CanonicalEvent::NewEvent
>;
```

### Passo 5 — Atualizar o wrapper Python (opcional)

```python
# src/ontology/inference_engine.py
from .events import NewEvent

_EVENT_CLASS_MAP = {
    # ... eventos existentes ...
    "new_event": NewEvent,
}

# em _result_to_event, se necessário:
elif event_cls is NewEvent:
    kwargs["new_field"] = result.depth_md
```

### Passo 6 — Recompilar e testar

```bash
pip install -e .
pytest tests/test_inference_engine.py -v
```

A abordagem com templates variádicos e `if constexpr` garante que a adição de um novo evento seja verificada em tempo de compilação: esquecer de registrar um ramo na `execute_rule` resulta em erro de compilação quando o evento é incluído em `CanonicalInferenceEngine`.

In [ ]:
# Demonstração conceitual de extensão em Python
from dataclasses import dataclass

@dataclass
class NewEvent(Event):
    new_field: float = None
    def __init__(self, new_field=None, **kwargs):
        kwargs.setdefault("event_type", "NewEventDetected")
        kwargs.setdefault("name", "New Event")
        super().__init__(**kwargs)
        self.new_field = new_field

# Simulação de conversão estendida
def extended_result_to_event(result, event_class_map_extra=None):
    event_cls = (event_class_map_extra or {}).get(result.event_type, Event)
    kwargs = {
        "event_type": result.event_type,
        "name": result.event_label_pt or result.event_label_en,
        "depth_md": result.depth_md,
        "confidence": result.confidence,
        "severity": result.severity,
        "recommendation": result.recommendation,
    }
    if event_cls is NewEvent:
        kwargs["new_field"] = result.depth_md
    return event_cls(**kwargs)

print("Extensão conceitual OK: nova classe NewEvent criada.")


## 9. Resumo

- O **InferenceEngine** transforma dados DTS/DAS em eventos operacionais estruturados.
- A implementação em **C++20** usa `CanonicalEvent`, `EventTraits`, `ResultGenerator`, concepts, `if constexpr` e fold expressions.
- Os **helpers numéricos** (`temporal_mean`, `remove_polynomial_baseline`, `remove_median_baseline`, `adaptive_threshold`, `find_peaks`, `find_valleys`, `das_energy_profile`) fornecem a base para as heurísticas.
- As regras de **Joule-Thomson**, **GLV Bellow Rupture** e **Cement Bond Evaluation** ilustram três tipos distintos de análise: contraste térmico, análise de sequência acústica e dispersão espacial.
- A **integração Python** expõe o motor via pybind11 e converte os resultados para a ontologia Alakoro.
- A extensão segue um padrão mecânico de 4–6 passos, com validação em tempo de compilação graças aos templates.